# Round-Trip Serialization

Demonstrates every serialization path and verifies fidelity.

| Format | To | From | Notes |
|---|---|---|---|
| Dict / JSON | `to_dict()` / `to_json()` | `from_dict()` / `from_json()` | Canonical lossless (thresholds + speech). |
| Long rows | `to_long_rows()` | `from_long_rows()` | Tidy threshold observations. |
| Long rows + speech | `to_long_rows(include_speech=True)` | `from_long_rows()` | Combined with measure_type discriminator. |
| Speech rows | `to_speech_rows()` | — | Tidy speech observations. |
| Wide row | `to_wide_row()` | `from_wide_row()` | Flat tabular, full fidelity. |
| Table rows | `to_table_rows()` | — | Normalized two-table output. |

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from audiogram_object import ThresholdPoint, EarAudiogram, BinauralAudiogram
from pprint import pprint

## Build a rich BinauralAudiogram

We include air + bone, masked bone, and NR — the full clinical picture.
Every field needs to survive every round-trip.

In [2]:
original = BinauralAudiogram(
    left=EarAudiogram(
        air={
            500:  ThresholdPoint(25.0),
            1000: ThresholdPoint(30.0),
            2000: ThresholdPoint(35.0),
            4000: ThresholdPoint(50.0),
            8000: ThresholdPoint(65.0),
        },
        bone={
            1000: ThresholdPoint(20.0, masked=True),
            2000: ThresholdPoint(25.0, masked=True),
            4000: ThresholdPoint(45.0, masked=True),
        },
    ),
    right=EarAudiogram(
        air={
            500:  ThresholdPoint(20.0),
            1000: ThresholdPoint(25.0),
            2000: ThresholdPoint(40.0),
            4000: ThresholdPoint(120.0, nr=True),
            8000: ThresholdPoint(120.0, nr=True),
        },
        bone={
            1000: ThresholdPoint(15.0, masked=True),
            2000: ThresholdPoint(30.0, masked=True),
        },
    ),
    audiogram_id='aud-rt-001',
    subject_id='pt-042',
    performed_at='2024-06-15',
    source='clinic',
)

print('Built BinauralAudiogram')
print('  Left air freqs:', original.left.available_frequencies())
print('  Left bone freqs:', original.left.available_frequencies('bone'))
print('  Right air freqs:', original.right.available_frequencies())
print('  Right bone freqs:', original.right.available_frequencies('bone'))

Built BinauralAudiogram
  Left air freqs: [500, 1000, 2000, 4000, 8000]
  Left bone freqs: [1000, 2000, 4000]
  Right air freqs: [500, 1000, 2000, 4000, 8000]
  Right bone freqs: [1000, 2000]


## Long rows

`to_long_rows()` is the canonical export: one row per threshold observation.
This is what feeds a DataFrame, a Django ORM bulk-insert, or a SQL table.

In [3]:
rows = original.to_long_rows()

print(f'Total rows: {len(rows)}')
print(f'Pathways present: {set(r["pathway"] for r in rows)}')
print(f'Masked rows: {[(r["ear"], r["freq_hz"], r["pathway"]) for r in rows if r["masked"]]}')
print(f'NR rows: {[(r["ear"], r["freq_hz"]) for r in rows if r["nr"]]}')
print()
pprint(rows[:3])

Total rows: 15
Pathways present: {'air', 'bone'}
Masked rows: [('left', 1000, 'bone'), ('left', 2000, 'bone'), ('left', 4000, 'bone'), ('right', 1000, 'bone'), ('right', 2000, 'bone')]
NR rows: [('right', 4000), ('right', 8000)]

[{'audiogram_id': 'aud-rt-001',
  'ear': 'left',
  'freq_hz': 500,
  'masked': False,
  'nr': False,
  'pathway': 'air',
  'performed_at': '2024-06-15',
  'source': 'clinic',
  'subject_id': 'pt-042',
  'threshold_db': 25.0},
 {'audiogram_id': 'aud-rt-001',
  'ear': 'left',
  'freq_hz': 1000,
  'masked': False,
  'nr': False,
  'pathway': 'air',
  'performed_at': '2024-06-15',
  'source': 'clinic',
  'subject_id': 'pt-042',
  'threshold_db': 30.0},
 {'audiogram_id': 'aud-rt-001',
  'ear': 'left',
  'freq_hz': 2000,
  'masked': False,
  'nr': False,
  'pathway': 'air',
  'performed_at': '2024-06-15',
  'source': 'clinic',
  'subject_id': 'pt-042',
  'threshold_db': 35.0}]


In [8]:
# One-liner to DataFrame — works with pandas or polars
try:
    import polars as pl
    df=pl.DataFrame(rows)
except ImportError:
    import pandas as pd
    df=pd.DataFrame(rows)

df

audiogram_id,subject_id,performed_at,source,ear,freq_hz,threshold_db,pathway,masked,nr
str,str,str,str,str,i64,f64,str,bool,bool
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""left""",500,25.0,"""air""",false,false
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""left""",1000,30.0,"""air""",false,false
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""left""",2000,35.0,"""air""",false,false
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""left""",4000,50.0,"""air""",false,false
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""left""",8000,65.0,"""air""",false,false
…,…,…,…,…,…,…,…,…,…
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""right""",2000,40.0,"""air""",false,false
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""right""",4000,120.0,"""air""",false,true
"""aud-rt-001""","""pt-042""","""2024-06-15""","""clinic""","""right""",8000,120.0,"""air""",false,true


In [10]:
# Round-trip
restored_rows = BinauralAudiogram.from_long_rows(rows)

# Spot-check every field that matters
assert restored_rows.audiogram_id == original.audiogram_id
assert restored_rows.subject_id == original.subject_id
assert restored_rows.performed_at == original.performed_at

# Air thresholds
assert restored_rows.left.air[500].threshold_db == 25.0
assert restored_rows.left.air[500].masked is False
assert restored_rows.left.air[500].nr is False

# Masked bone
assert restored_rows.left.bone[1000].threshold_db == 20.0
assert restored_rows.left.bone[1000].masked is True

# NR
assert restored_rows.right.air[4000].threshold_db == 120.0
assert restored_rows.right.air[4000].nr is True

# All frequencies preserved
assert restored_rows.left.available_frequencies() == original.left.available_frequencies()
assert restored_rows.right.available_frequencies('bone') == original.right.available_frequencies('bone')

print('Long rows round-trip: OK')

Long rows round-trip: OK


## Table rows

`to_table_rows()` returns a normalized pair: one `tests` row + many `observations` rows.
This mirrors the Django ORM two-table structure.

In [11]:
test_row, obs_rows = original.to_table_rows()

print('Test row:')
pprint(test_row)
print()
print(f'Observation rows: {len(obs_rows)}')
print('First obs row:')
pprint(obs_rows[0])

Test row:
{'audiogram_id': 'aud-rt-001',
 'meta': {},
 'performed_at': '2024-06-15',
 'schema_version': '1.0',
 'source': 'clinic',
 'subject_id': 'pt-042'}

Observation rows: 15
First obs row:
{'audiogram_id': 'aud-rt-001',
 'ear': 'left',
 'freq_hz': 500,
 'masked': False,
 'nr': False,
 'pathway': 'air',
 'threshold_db': 25.0}


## Dict round-trip

In [12]:
d = original.to_dict()
pprint(d)

{'audiogram_id': 'aud-rt-001',
 'left': {'air': {500: {'masked': False, 'nr': False, 'threshold_db': 25.0},
                  1000: {'masked': False, 'nr': False, 'threshold_db': 30.0},
                  2000: {'masked': False, 'nr': False, 'threshold_db': 35.0},
                  4000: {'masked': False, 'nr': False, 'threshold_db': 50.0},
                  8000: {'masked': False, 'nr': False, 'threshold_db': 65.0}},
          'bone': {1000: {'masked': True, 'nr': False, 'threshold_db': 20.0},
                   2000: {'masked': True, 'nr': False, 'threshold_db': 25.0},
                   4000: {'masked': True, 'nr': False, 'threshold_db': 45.0}}},
 'meta': {},
 'performed_at': '2024-06-15',
 'right': {'air': {500: {'masked': False, 'nr': False, 'threshold_db': 20.0},
                   1000: {'masked': False, 'nr': False, 'threshold_db': 25.0},
                   2000: {'masked': False, 'nr': False, 'threshold_db': 40.0},
                   4000: {'masked': False, 'nr': True, 'thresho

In [13]:
restored_dict = BinauralAudiogram.from_dict(d)

assert restored_dict.audiogram_id == original.audiogram_id
assert restored_dict.left.air[1000].threshold_db == original.left.air[1000].threshold_db
assert restored_dict.left.bone[2000].masked == original.left.bone[2000].masked
assert restored_dict.right.air[4000].nr == original.right.air[4000].nr
assert restored_dict.right.air[4000].threshold_db == original.right.air[4000].threshold_db

print('Dict round-trip: OK')

Dict round-trip: OK


## JSON round-trip

In [14]:
import json

json_str = original.to_json()
print(json.dumps(json.loads(json_str), indent=2))

{
  "audiogram_id": "aud-rt-001",
  "subject_id": "pt-042",
  "performed_at": "2024-06-15",
  "source": "clinic",
  "meta": {},
  "left": {
    "air": {
      "500": {
        "threshold_db": 25.0,
        "masked": false,
        "nr": false
      },
      "1000": {
        "threshold_db": 30.0,
        "masked": false,
        "nr": false
      },
      "2000": {
        "threshold_db": 35.0,
        "masked": false,
        "nr": false
      },
      "4000": {
        "threshold_db": 50.0,
        "masked": false,
        "nr": false
      },
      "8000": {
        "threshold_db": 65.0,
        "masked": false,
        "nr": false
      }
    },
    "bone": {
      "1000": {
        "threshold_db": 20.0,
        "masked": true,
        "nr": false
      },
      "2000": {
        "threshold_db": 25.0,
        "masked": true,
        "nr": false
      },
      "4000": {
        "threshold_db": 45.0,
        "masked": true,
        "nr": false
      }
    }
  },
  "right": {
    "air

In [15]:
restored_json = BinauralAudiogram.from_json(json_str)

assert restored_json.audiogram_id == original.audiogram_id
assert restored_json.subject_id == original.subject_id
assert restored_json.left.bone[1000].masked is True
assert restored_json.right.air[8000].nr is True

print('JSON round-trip: OK')

JSON round-trip: OK


## Summary

| Format | Thresholds | Speech | Lossless |
|---|---|---|---|
| Dict / JSON | yes | yes | yes |
| Wide row | yes | yes | yes |
| Long rows (default) | yes | no | thresholds only |
| Long rows (`include_speech=True`) | yes | yes | yes |
| Speech rows | no | yes | speech only |
| Table rows | yes | no | thresholds only |

Dict/JSON is the canonical lossless interchange format.